# OpenAlex Enrichment

Matches publication titles or DOIs against the [OpenAlex API](https://openalex.org/) (free, no key required) and adds `Citation_Count#number`, `OA_Status`, and `Fields_of_Study#multi` columns.

Intended for bibliography or publication surveys. Rows without a title or DOI match are left blank.

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        from IPython.display import HTML as _HTML
        display(_HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE parameters found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open this notebook via SuAVEDispatch from your survey in SuAVE.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
else:
    # Colab: try Drive/cache silently; credentials cell below handles fallback
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: verify session loaded from Drive/cache --------------------------
from IPython.display import HTML
if su.ENV.colab:
    if _p is None:
        display(HTML(
            '<div style="background:#dc3545;color:white;padding:14px 16px;'
            'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
            '&#9888; No SuAVE session found.'
            '<br><span style="font-size:12px;font-weight:normal">'
            'Open SuAVEDispatch from the correct survey in SuAVE, then re-open this notebook.'
            '</span></div>'))
        class _Stop(Exception):
            def _render_traceback_(self): return []
        raise _Stop()
    display(HTML(
        '<div style="font-size:12px;padding:8px 12px;border-radius:4px;margin:3px 0;'
        'background:#e8f5e9;border:1px solid #81c784">'
        'Session loaded &mdash; survey <b>' + _p.get('survey', '?') + '</b>'
        ', user <b>' + _p.get('user', '?') + '</b>.'
        '<br><span style="color:#666;font-size:11px">'
        'Wrong survey? Go back to SuAVE, open the correct survey, and click Jupyter again.'
        '</span></div>'))
else:
    su._skipped('Colab only')


In [ ]:
# ── Variables used throughout this notebook ────────────────────────────────
if _p is None:
    from IPython.display import HTML as _HTML
    display(_HTML(
        '<div style="background:#dc3545;color:white;padding:14px 16px;'
        'border-radius:6px;font-size:15px;font-weight:bold;line-height:1.6;margin:4px 0">'
        '&#9888; Parameters not loaded.'
        '<br><span style="font-size:12px;font-weight:normal">'
        'Navigate to your survey in SuAVE and relaunch this notebook to load parameters.'
        '</span></div>'))
    class _Stop(Exception):
        def _render_traceback_(self): return []
    raise _Stop()
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import time

## 1. Load data and select identifier column

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

id_col = widgets.Dropdown(
    options=text_cols, description='Title or DOI column:',
    layout=widgets.Layout(width='60%')
)
match_by = widgets.RadioButtons(
    options=['Title (fuzzy search)', 'DOI (exact match)'],
    description='Match by:'
)
email_hint = widgets.Text(
    value='', placeholder='your@email.com (polite pool, faster)',
    description='Email (optional):', layout=widgets.Layout(width='60%')
)
display(id_col, match_by, email_hint)

## 2. Fetch from OpenAlex

In [ ]:
OA_BASE = 'https://api.openalex.org/works'

col        = id_col.value
by_doi     = 'DOI' in match_by.value
email      = email_hint.value.strip()
headers    = {'User-Agent': f'suave-notebooks (+{email})'} if email else {'User-Agent': 'suave-notebooks'}
from tqdm.auto import tqdm

_cache = {}

def _lookup(value):
    if value in _cache:
        return _cache[value]
    try:
        time.sleep(0.12)  # ~8 req/sec, well within OA limits
        if by_doi:
            doi = str(value).strip().replace('https://doi.org/', '')
            resp = _req.get(f"{OA_BASE}/doi:{doi}", headers=headers, timeout=10)
            if resp.status_code == 200:
                work = resp.json()
            else:
                _cache[value] = (None, None, None); return None, None, None
        else:
            resp = _req.get(OA_BASE, params={
                'search': str(value)[:200], 'per-page': 1,
                'select': 'id,title,cited_by_count,open_access,topics'
            }, headers=headers, timeout=10)
            results = resp.json().get('results', [])
            if not results:
                _cache[value] = (None, None, None); return None, None, None
            work = results[0]

        citations  = work.get('cited_by_count')
        oa_status  = (work.get('open_access') or {}).get('oa_status', None)
        topics     = [t['display_name'] for t in (work.get('topics') or [])[:5]]
        fields_str = '|'.join(topics) if topics else None
        _cache[value] = (citations, oa_status, fields_str)
        return citations, oa_status, fields_str
    except Exception:
        _cache[value] = (None, None, None)
        return None, None, None

cite_counts, oa_statuses, fields = [], [], []

for val in tqdm(df[col], desc='OpenAlex lookup'):
    if not val or pd.isna(val) or str(val).strip() == '':
        cite_counts.append(None); oa_statuses.append(None); fields.append(None)
        continue
    c, o, f = _lookup(str(val).strip())
    cite_counts.append(c); oa_statuses.append(o); fields.append(f)

df['Citation_Count#number'] = cite_counts
df['OA_Status']             = oa_statuses
df['Fields_of_Study#multi'] = fields

n_ok = sum(1 for c in cite_counts if c is not None)
printmd(f"**Matched {n_ok}/{len(df)} rows in OpenAlex.**")
display(df[[col, 'Citation_Count#number', 'OA_Status', 'Fields_of_Study#multi']].dropna(subset=['Citation_Count#number']).head(10))

## 3. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)


In [ ]:
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)
